
# NLP Lab Programs — Questions 14 to 18

This notebook contains simple, exam-friendly Python implementations for:

1. **Viterbi Algorithm** for POS tagging
2. **Bigram calculation and sentence probability**
3. **TF-IDF matrix and cosine similarity**
4. **PPMI matrix and cosine similarity**
5. **Naive Bayes Word-Sense Disambiguation (WSD)** using Bag-of-Words and Add-1 smoothing


## 14. Viterbi Algorithm for POS Tagging

In [ ]:

# Viterbi Algorithm for POS Tagging

states = ["N", "V", "D"]  # Noun, Verb, Determiner

start_prob = {
    "N": 0.3,
    "V": 0.2,
    "D": 0.5
}

transition_prob = {
    "N": {"N": 0.4, "V": 0.4, "D": 0.2},
    "V": {"N": 0.5, "V": 0.1, "D": 0.4},
    "D": {"N": 0.8, "V": 0.1, "D": 0.1}
}

emission_prob = {
    "the": {"N": 0.05, "V": 0.05, "D": 0.90},
    "dog": {"N": 0.80, "V": 0.10, "D": 0.10},
    "runs": {"N": 0.10, "V": 0.80, "D": 0.10}
}

def viterbi(sentence, states, start_prob, transition_prob, emission_prob):
    words = sentence.lower().split()
    V = [{}]
    backpointer = [{}]

    # Initialization
    for state in states:
        V[0][state] = (
            start_prob[state] *
            emission_prob.get(words[0], {}).get(state, 0.01)
        )
        backpointer[0][state] = None

    # Recursion
    for t in range(1, len(words)):
        V.append({})
        backpointer.append({})

        for state in states:
            emission = emission_prob.get(words[t], {}).get(state, 0.01)

            best_previous = max(
                states,
                key=lambda prev: V[t-1][prev] * transition_prob[prev][state]
            )

            V[t][state] = (
                V[t-1][best_previous]
                * transition_prob[best_previous][state]
                * emission
            )
            backpointer[t][state] = best_previous

    # Termination
    best_last = max(states, key=lambda state: V[-1][state])

    best_path = [best_last]
    for t in range(len(words) - 1, 0, -1):
        best_path.append(backpointer[t][best_path[-1]])

    best_path.reverse()
    return list(zip(words, best_path)), max(V[-1].values())


sentence = "the dog runs"
tags, probability = viterbi(
    sentence, states, start_prob, transition_prob, emission_prob
)

print("Sentence:", sentence)
print("Most probable POS sequence:")
print(tags)
print("Probability:", probability)


## 15. Bigrams and Sentence Probability

In [ ]:

# Calculate bigrams from a corpus and sentence probability

from collections import Counter

corpus = [
    "I love natural language processing",
    "I love machine learning",
    "natural language processing is interesting"
]

def get_bigrams(corpus):
    bigrams = Counter()
    unigrams = Counter()

    for sentence in corpus:
        words = ["<s>"] + sentence.lower().split() + ["</s>"]

        for word in words:
            unigrams[word] += 1

        for i in range(len(words) - 1):
            bigrams[(words[i], words[i + 1])] += 1

    return unigrams, bigrams

unigrams, bigrams = get_bigrams(corpus)

print("Bigrams:")
for bg, count in bigrams.items():
    print(bg, ":", count)

def sentence_probability(sentence, unigrams, bigrams):
    words = ["<s>"] + sentence.lower().split() + ["</s>"]
    probability = 1.0

    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i + 1]

        # P(w2 | w1) = Count(w1,w2) / Count(w1)
        count_bigram = bigrams.get((w1, w2), 0)
        count_unigram = unigrams.get(w1, 0)

        if count_unigram == 0 or count_bigram == 0:
            return 0.0

        probability *= count_bigram / count_unigram

    return probability

test_sentence = "I love natural language processing"
print("\nProbability of:", test_sentence)
print(sentence_probability(test_sentence, unigrams, bigrams))


## 16. TF-IDF Matrix and Cosine Similarity

In [ ]:

# TF-IDF matrix and cosine similarity

import math
import re
from collections import Counter

documents = [
    "I love machine learning",
    "I love natural language processing",
    "machine learning is useful"
]

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

tokenized_docs = [tokenize(doc) for doc in documents]
vocabulary = sorted(set(word for doc in tokenized_docs for word in doc))

N = len(documents)

# Term Frequency
tf = []
for doc in tokenized_docs:
    counts = Counter(doc)
    total = len(doc)
    tf.append({
        word: counts[word] / total
        for word in vocabulary
    })

# Inverse Document Frequency
idf = {}
for word in vocabulary:
    df = sum(1 for doc in tokenized_docs if word in doc)
    idf[word] = math.log(N / df)

# TF-IDF
tfidf = []
for doc_tf in tf:
    tfidf.append([
        doc_tf[word] * idf[word]
        for word in vocabulary
    ])

print("Vocabulary:")
print(vocabulary)

print("\nTF-IDF Matrix:")
for row in tfidf:
    print([round(x, 4) for x in row])

def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot / (norm1 * norm2)

print("\nCosine similarity between Document 1 and Document 2:")
print(round(cosine_similarity(tfidf[0], tfidf[1]), 4))

# Word similarity using word columns of the TF-IDF matrix
def word_similarity(word1, word2):
    if word1 not in vocabulary or word2 not in vocabulary:
        return 0.0

    i = vocabulary.index(word1)
    j = vocabulary.index(word2)

    vector1 = [tfidf[d][i] for d in range(N)]
    vector2 = [tfidf[d][j] for d in range(N)]

    return cosine_similarity(vector1, vector2)

print("\nCosine similarity between words 'machine' and 'learning':")
print(round(word_similarity("machine", "learning"), 4))


## 17. PPMI Matrix and Cosine Similarity

In [ ]:

# PPMI (Positive Pointwise Mutual Information) matrix

import math
import re
from collections import Counter

documents = [
    "I love machine learning",
    "I love natural language processing",
    "machine learning is useful"
]

tokenized_docs = [re.findall(r"\b[a-z]+\b", d.lower()) for d in documents]

# Build word-context co-occurrence counts.
# Here, each document is treated as the context.
words = sorted(set(w for doc in tokenized_docs for w in doc))

word_doc_count = Counter()
for doc in tokenized_docs:
    for word in set(doc):
        word_doc_count[word] += 1

total_pairs = sum(word_doc_count.values())

# Co-occurrence matrix: word x document
cooccurrence = {
    word: [1 if word in doc else 0 for doc in tokenized_docs]
    for word in words
}

# PPMI(word, document) = max(PMI, 0)
# PMI = log2(P(word,document)/(P(word)*P(document)))
ppmi = {}

for word in words:
    ppmi[word] = []

    word_count = word_doc_count[word]

    for d in range(len(documents)):
        pair_count = cooccurrence[word][d]

        if pair_count == 0:
            ppmi[word].append(0.0)
            continue

        document_words = len(set(tokenized_docs[d]))

        p_word = word_count / total_pairs
        p_doc = document_words / total_pairs
        p_word_doc = pair_count / total_pairs

        pmi = math.log2(p_word_doc / (p_word * p_doc))
        ppmi[word].append(max(pmi, 0))

print("Vocabulary:")
print(words)

print("\nPPMI Matrix (rows = words, columns = documents):")
for word in words:
    print(word, [round(x, 4) for x in ppmi[word]])

def cosine(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    n1 = math.sqrt(sum(a * a for a in v1))
    n2 = math.sqrt(sum(b * b for b in v2))

    if n1 == 0 or n2 == 0:
        return 0.0

    return dot / (n1 * n2)

def ppmi_word_similarity(word1, word2):
    if word1 not in ppmi or word2 not in ppmi:
        return 0.0
    return cosine(ppmi[word1], ppmi[word2])

print("\nCosine similarity between words 'machine' and 'learning':")
print(round(ppmi_word_similarity("machine", "learning"), 4))

# Document vectors can be obtained by transposing the PPMI matrix
document_vectors = []
for d in range(len(documents)):
    document_vectors.append([ppmi[word][d] for word in words])

print("\nCosine similarity between Document 1 and Document 2:")
print(round(cosine(document_vectors[0], document_vectors[1]), 4))


## 18. Naive Bayes Word-Sense Disambiguation

In [ ]:

# Naive Bayes classifier with Add-1 smoothing
# Bag-of-Words features for disambiguating the word "bass"

training_data = [
    ("I love fish. The smoked bass fish was delicious.", "fish"),
    ("The bass fish swam along the line.", "fish"),
    ("He hauled in a big catch of smoked bass fish.", "fish"),
    ("The bass guitar player played a smooth jazz line.", "guitar")
]

test_sentence = (
    "He loves jazz. The bass line provided the foundation "
    "for the guitar solo in the jazz piece"
)

test_word = "bass"

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

# Remove the ambiguous word itself from the features.
def features(text, ambiguous_word):
    return [w for w in tokenize(text) if w != ambiguous_word.lower()]

# Build vocabulary
vocabulary = sorted(set(
    word
    for sentence, sense in training_data
    for word in features(sentence, test_word)
))

classes = sorted(set(sense for sentence, sense in training_data))

# Count documents for each sense
class_counts = Counter(sense for sentence, sense in training_data)
total_documents = len(training_data)

# Word counts for each class
word_counts = {
    sense: Counter()
    for sense in classes
}

total_words = {
    sense: 0
    for sense in classes
}

for sentence, sense in training_data:
    words = features(sentence, test_word)
    word_counts[sense].update(words)
    total_words[sense] += len(words)

def predict(sentence):
    test_words = features(sentence, test_word)
    scores = {}

    for sense in classes:
        # Prior probability P(class)
        score = class_counts[sense] / total_documents

        # Naive Bayes: multiply P(word | class)
        denominator = total_words[sense] + len(vocabulary)

        for word in test_words:
            # Add-1 smoothing
            probability = (
                word_counts[sense][word] + 1
            ) / denominator

            score *= probability

        scores[sense] = score

    return max(scores, key=scores.get), scores

prediction, scores = predict(test_sentence)

print("Test Sentence:")
print(test_sentence)

print("\nTest Word:", test_word)

print("\nClass probabilities:")
for sense, score in scores.items():
    print(sense, ":", score)

print("\nPredicted Sense:", prediction)



### Expected result for Question 18

For the supplied training examples and test sentence, the predicted sense is:

**guitar**

The words such as **jazz, line, guitar, solo, foundation, piece** provide stronger evidence for the musical sense of *bass* than the fish sense.



## Quick Exam Notes

### Viterbi
- Initialization: calculate the probability for the first word.
- Recursion: choose the previous tag giving the maximum probability.
- Termination: choose the tag with maximum final probability.
- Backtrack to obtain the best tag sequence.

### Bigram
\[
P(w_i|w_{i-1}) = \frac{Count(w_{i-1},w_i)}{Count(w_{i-1})}
\]

Sentence probability:

\[
P(W) = \prod_i P(w_i|w_{i-1})
\]

### TF-IDF
\[
TF(t,d)=\frac{count(t,d)}{\text{number of terms in }d}
\]

\[
IDF(t)=\log\left(\frac{N}{DF(t)}\right)
\]

\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

### PPMI
\[
PMI(w,c)=\log_2\frac{P(w,c)}{P(w)P(c)}
\]

\[
PPMI(w,c)=\max(PMI(w,c),0)
\]

### Naive Bayes with Add-1 Smoothing
\[
P(w|c)=\frac{Count(w,c)+1}{TotalWords(c)+|V|}
\]

For classification:

\[
P(c|d)\propto P(c)\prod_i P(w_i|c)
\]
